# 📚 LangChain ile Retrieval Augmented Generation'a Giriş 🦜🔗

Bu notebook'ta LangChain kullanarak Retrieval Augmented Generation'ı nasıl kullanacağınızı öğreneceksiniz.

Kendi dokümanlarımız hakkında soruları cevaplamak için bir LLM kullanacağız!

## ⚙️ Kurulum

👉 Temel kütüphaneleri içe aktarmak için aşağıdaki hücreyi çalıştırın.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

👉 API anahtarımızı tekrar yüklemek için aşağıdaki hücreyi çalıştırın:

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

## 📚 Neden RAG?

Bir LLM tek başına öğrendiği her şey hakkında sorulara yanıt verebilir.

Bunun birkaç dezavantajı vardır:
- Eğitim verisi geçmişten gelir ve en güncel verilerle güncellenmez.
- Sadece üzerinde eğitildiği veriyi bilir.

Kendi verilerimizle çalışmak için bir LLM kullanmak istiyoruz. İşte RAG veya Retrieval-Augmented Generation burada devreye girer.

1. **Retrieval-Augmented Generation (RAG)**, gerçeklik doğruluğunu artırmak için bir dil modeli ile bir döküman alıcısını birleştirir.
2. **Yanıt üretmeden önce ilgili harici dökümanları alır** (örneğin, bir bilgi tabanından).
3. **Dil modeli hem prompt hem de alınan bağlamı kullanır** ve daha bilgili ve temelli çıktılar üretir.

## 🇪🇺 Bağlam

Bu zorlukta, Avrupa Parlamentosu'ndan gelen dökümanlarla çalışacağız.

Bir gazeteci olduğunuzu ve Avrupa Parlamentosu'nun genel kurul oturumları sırasında belirli bir konu hakkında neler söylendiğini bilmek istediğinizi hayal edin. Bu oturumlar yılda 12 kez Strasbourg'da gerçekleşir ve 4 gün sürer. Oturumların transkriptleri AP'nin web sitesinde mevcuttur.

Kesinlikle tüm bu transkriptleri gözden geçirmek istemezsiniz. O halde hayatımızı kolaylaştırmak için RAG'den faydalanalım!

Bu çalışmak için iyi bir veri çünkü her zaman test etmek için yepyeni veriler alabiliriz.

## 📘 Veriyi alalım

1. [AP'nin web sitesine](https://www.europarl.europa.eu/plenary/en/debates-video.html) gidin. 
1. Bu sizi en son genel kurul oturumuna götürecek.
1. İlk tarihin altında, "▶️ Verbatim reports HTML" bölümünde `HTML`'e tıklayın.
1. Sayfanın alt kısmına kaydırın ve alttaki PDF dosyasını indirin.
1. Dosyayı `data` klasörüne kaydedin.

Bir dökümanla başlayacağız, ancak daha sonra kullanmak üzere birkaç başka günün de aynısını indirebilirsiniz.

Dökümana göz atın. Kaç sayfası var? Fikir edinmek için dökümanda hızlıca gezinin.


## 🔢 Dökümanları gömme (embedding)

Dökümanları gömmek, tüm dökümanları veya döküman parçalarını vektörlere çevireceğimiz anlamına gelir.

LangChain🦜🔗 yine çok yardımcı olacak.

Bir embedder örneği oluşturalım ve deneyelim. LLM olarak Gemini kullandığımız için Google'ın metin embedderlarını kullanalım.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

👉 Basit bir metin parçasını gömmek için embedder'ın `.embed_query()` metodunu deneyin.

In [ ]:
# Embed a text like "What is the capital of France?" and save it to a variable `sample_embedding`
sample_embedding = embeddings.embed_query("What is the capital of France?")

👉 Bu `sample_embedding`'i keşfetmek için zaman ayırın. Neye benziyor? Türü nedir? Embedding boyutu nedir?

In [ ]:
print(type(sample_embedding))
print(len(sample_embedding))

## 💾 Gerçek verimizi PDF'den yükleyelim

Artık bir embedding'in neye benzediğini bildiğimize göre, gerçek verimizle çalışma zamanı.

👉 [LangChain dokümantasyonuna](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs) gidin ve PyPDF kullanarak bir PDF'i nasıl yükleyebileceğinizi öğrenin.

👉 Sonra devam edin ve daha önce indirdiğiniz PDF'lerden birini yükleyin.

In [ ]:
!curl https://www.europarl.europa.eu/doceo/document/CRE-10-2025-05-05_EN.pdf > data/CRE-10-2025-05-05_EN.pdf
!curl https://www.europarl.europa.eu/doceo/document/CRE-10-2025-05-06_EN.pdf > data/CRE-10-2025-05-06_EN.pdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "data/CRE-10-2025-05-05_EN.pdf"

loader = PyPDFLoader(file_path)

pages = loader.load()

👉 `pages`'i keşfedin:
- Veri türü nedir?
- Kaç sayfanız var?
- Bir sayfanın türü nedir?
- Bir sayfanın içeriğine nasıl erişebilirsiniz?
- Tam döküman kaç karakter içeriyor?
- Bir sayfanın `metadata`'sında neler var?

In [ ]:
print("Type of pages:          ", type(pages))
print("Nb of pages:            ", len(pages))
print("Type of one page:       ", type(pages[0]))
print("Total nb of characters: ", sum([len(page.page_content) for page in pages]))

In [ ]:
Markdown(pages[40].page_content)

In [ ]:
pages[40].metadata

## ✂️ Verimizi bölelim

Tam dökümanımız embed edilmek için çok uzun. Metin embedder'ımız 2.048 token'a kadar girdi alabilir. Gemini modelleri için bu yaklaşık 8.196 karakter demektir (token başına 4 karakter).

Bu yüzden dökümanımızı daha küçük parçalara bölmek istiyoruz.

Zaten çalışabiliriz bir dizi sayfamız var. Ancak sayfa sonları biraz keyfidır: genellikle bir cümlenin ortasında görünürler.

Ayrıca sayfalar arasında örtüşme yoktur. Bu yüzden bir sayfanın ilk satırı önceki tüm bağlamı kaybeder. Biraz örtüşmeyle tam metni bölmek daha iyidir.

İlk olarak, PDF'i bu sefer bölmeden tekrar yükleyeceğiz.

In [ ]:
loader = PyPDFLoader(file_path, mode='single')
pdf = loader.load()
pdf_text = pdf[0].page_content
len(pdf_text)

Artık tüm PDF'ımızı tek bir döküman olarak elde ettiğimize göre, onu daha akıllı bir şekilde parçalara bölebiliriz.

👉 Tekrar, [LangChain'in "Recursive olarak bölme" dokümantasyonuna](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) gidin ve `pdf` _dökümanlarımızı_ parçalara (LangChain'de `documents` olarak adlandırılır) nasıl bölebileceğinizi öğrenin.

2_000 karakter parçalar halinde (bizim durumumuzda bu yaklaşık yarım sayfa) 400'lük bir örtüşmeyle bölün. İsterseniz başka değerlerle de deneyebilirsiniz.

`RecursiveCharacterTextSplitter`'ın `.split_documents()` metodunu kullanın: bu metod girdi olarak bir döküman alır ve bölünmüş dökümanlar çıktısı verir.

In [ ]:
# Import the necessary libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter instance. Start with a chunk size of 1000 characters and an overlap of 200 characters.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2_000,      # chunk size (characters)
    chunk_overlap=400,     # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

# Split `pages` into smaller chunks. Store in a variable called `all_splits`.
all_splits = text_splitter.split_documents(pdf)

# Print the number of splits
print(f"Split transcript into {len(all_splits)} sub-documents.")

👉 `all_splits`'i inceleyin:
- Veri türü nedir?
- Kaç bölümünüz var?
- Bir bölümün türü nedir?
- Bir bölümün içeriğine nasıl erişebilirsiniz?
- Toplamda kaç karakterimiz var şimdi?
- Bir bölümün `metadata`'sında neler var?

In [ ]:
print("Type of all_splits:          ", type(all_splits))
print("Nb of all_splits:            ", len(all_splits))
print("Type of one all_split:       ", type(all_splits[0]))
print("Total nb of characters:      ", sum([len(all_split.page_content) for all_split in all_splits]))

In [ ]:
Markdown(all_splits[80].page_content)

In [ ]:
all_splits[80].metadata

## 🗄️ Hepsini bir araya getirelim: dökümanlarımızı gömleyelim ve bir vektör deposunda saklayalım

Bizde var:
- Bir embedder
- Veri yüklemek için bir loader
- Dökümanımızı dökümanlara bölmek için bir metin bölücüsü

Eksik olan nedir?

Dökümanlarımızı gömleyebiliriz ama bunları bir yerde saklamak istiyoruz. Vektör deposu işte burada devreye giriyor: şunları kaydetmemizi sağlar:
- döküman (parça),
- onun embedding'i,
- onun metadata'sı.

Bir sonraki adımda dökümanları verimli bir şekilde alabiliriz.

👉 [LangChain'in "Vektör depoları" dokümantasyonunu](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin ve bir `InMemoryVectorStore` nasıl oluşturabileceğinizi görün.

In [ ]:
# Import the necessary libraries
from langchain_core.vectorstores import InMemoryVectorStore

# Create an in-memory vector store using the embedder `embeddings` we created earlier
vector_store = InMemoryVectorStore(embeddings)

# Add the `all_splits` to the vector store and store the result in a variable called `document_ids`
document_ids = vector_store.add_documents(documents=all_splits)

In [ ]:
# Have a look at the first 3 document IDs
print(document_ids[:3])

In [ ]:
# Use the vector store's `get_by_ids` method. You have to give it a list of document IDs.
vector_store.get_by_ids(document_ids[:3])

👉 Bir vektör deposunun dökümanının içeriğine ve metadata'sına nasıl erişebilirsiniz?

In [ ]:
one_doc = vector_store.get_by_ids(document_ids[:1])[0]
display(Markdown(one_doc.page_content))
display(one_doc.metadata)

## 🔎 Benzer dökümanları almak için vektör deposunu kullanın

Artık dökümanları gömlediğimize göre, benzer dökümanları almak için vektör deposunu kullanabiliriz.

👉 [LangChain'in "Vektör depoları" dokümantasyonunda](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) bunun nasıl çalıştığını kontrol edin.

Bir sorgu kullanın, örneğin "Tarım politikası hakkındaki tartışmayı özetleyin." ve en benzer dökümanları bulun. Ayrıca alacak döküman sayısını da belirtebilirsiniz.

In [ ]:
# Save your question into a variable called `query`
query = "Summarize the discussion on agricultural policy."

# Use the vector store to find similar documents to the query. Store the result in a variable called `retrieved_docs`
retrieved_docs = vector_store.similarity_search(query, k=6)

In [ ]:
for doc in retrieved_docs:
    display(Markdown(doc.page_content))

Bu, RAG'ın "Retrieval" (Alma) kısmını sonuçlandırır: artık sorgumuzla en benzer olan dökümanları bulabiliriz.

İşin çoğu artık tamamlandı!

## 💬 Soruğumuza bir cevap üretelim

Şimdiye kadar benzer dökümanları almamızı sağlayacak sadece bir **embedding modeli** kullandık.

şimdi soruğumuza bir cevap almak için üretken bir LLM kullanacağız: ona aldığımız dökümanları ve soruğumuzu besleyeceğiz.

Bunu yapmanın en ilkel yolu tüm girdilerimizi birleştirmek, soruğumuzu eklemek ve sonucu görmek olurdu.

Bir deneme yapalım.

👉 İlk olarak önceki zorluklarda olduğu gibi bir LLM örneği oluşturun.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

Sonra ilkel bir prompt oluşturun:

In [ ]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Şimdi prompt'ı kullanın:

In [ ]:
response = llm.invoke(prompt)
Markdown(response.content)

Fena değil, ama modele daha fazla rehberlik sağlayarak daha kapsamlı bir prompt yazarak daha iyisini yapabiliriz.

Bunu yapan ilk kişiler biz değiliz ve LangChain'de bizim için hazırlanmış prompt kütüphanesi var.

👉 Aşağıdaki hücreyi çalıştırın ve nasıl çalıştığını anlamaya çalışın. (LangSmithMissingAPIKeyWarning hakkında bir uyarı alacaksınız, bunu göz ardı edebilirsiniz.)

In [ ]:
from langchain_classic import hub

prompt_template = hub.pull("rlm/rag-prompt")

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("\n")
print(example_messages[0].content)

LangChain'in bizim için nasıl daha kesin bir prompt ürettidiğini gördünüz mü? Bunu RAG'imiz için kullanağım!

👉 İlk olarak, tüm alınan dökümanları iki satır sonu ile ayrılmış tek uzun string'e birleştirin.

In [ ]:
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
Markdown(docs_content)

👉 Sonra, sorgu ve alınan dökümanlarınızdan başlayarak bir `prompt` oluşturun. Yukarıdaki örneğe bakmayı unutmayın.

In [ ]:
prompt = prompt_template.invoke(
    {"context": docs_content, "question": query}
)

👉 Son olarak LLM modelini az önce oluşturduğumuz `the_prompt` ile kullanın:

In [ ]:
answer = llm.invoke(prompt)

In [ ]:
Markdown(answer.content)

🎉 İlk RAG'ımızı tamamladık: LLM kendisine sağladığımız dökümanlarda ***temelli*** metin üretti.

## 💾 Embedding'lerimizi kalıcılaştırma

Şimdiye kadar belleğin içinde bir vektör deposuyla çalıştık. Notebook'ı kapattığınızda tüm embedding'leri de kaybedeceksiniz.

⚠️ Bu embedding'lerin sağlayıcınızın platformunda, bu durumda Google'ın makinelerinde çalışan modeller tarafından üretildiğini unutmayın. Ve bunlar bedava çalışmıyor. 💰

Bu gibi küçük, nispeten küçük bir döküman için maliyet düşük ama hızla birikir. Şimdiye kadar sadece bir günün transkriptleri ile çalıştık. Oturum başına 3 tane daha, yılda 12 oturum, birkaç yıl var...

Bunu çözmek için sadece belleğin içindeki vektör depomuzu kalıcı olanla değiştireceğiz. LangChain'in avantajı bu: bileşenleri değiştirmek çok kolay.

Bellek içi vektör depomuz deneme için harikayı, şimdi onu başka biriyle değiştireceğiz. Çok popüler bir vektör deposu olan [Chroma](https://www.trychroma.com/)'yı kullanacağız. Yerel olarak çalıştırabiliriz ve LangChain aracılığıyla kullanabiliriz.

Tüm akışımızı yeniden oluşturacağız. Birkaç kod hücresinde her şeyi tekrar bir araya getirmeye çalışmak iyi bir alıştırmadır. Aynı zamanda her şeyi yeniden kullanılabilir koda dönüştürüeğiz.

Sonunda iki fonksiyonumuz olmasını istiyoruz:

1. `embed_and_store()`: Daha fazla verimiz olması için başka bir oturumun transkriptini vektör db'mize eklemek.
2. `answer()`: Vektör depomuzdan farklı sorularla sorgulama yapmak.

#### 1. Chroma vektör deposu örneği oluşturun

👉 **Veri kalıcılığı** (yani verileri diskteki bir dizinde depolamak) ile Chroma vektör deposunun nasıl oluşturulacağını görmek için [LangChain'in dokümantasyonuna](https://python.langchain.com/docs/integrations/vectorstores/chroma/) bakın.

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="ep_plenary",
    embedding_function=embeddings,
    persist_directory="./chroma_ep_follower",
)

#### 2. `embed_and_store()` oluşturun

👉 Bu fonksiyon için kodu tamamlayın:

In [ ]:
def embed_and_store(file_path, vector_store):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store."""
    # Load the PDF file
    # $CHALLENGIFY_BEGIN
    loader = PyPDFLoader(file_path, mode='single')
    pdf = loader.load()
    # $CHALLENGIFY_END

    # Split the pages into chunks
    # $CHALLENGIFY_BEGIN
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2_000,  # chunk size (characters)
        chunk_overlap=400,  # chunk overlap (characters)
        add_start_index=True,  # track index in original document
    )
    all_splits = text_splitter.split_documents(pdf)
    # $CHALLENGIFY_END

    # # Add the session_date to the metadata
    # for split in all_splits:
    #     split.metadata['session_date'] = session_date

    # Add the chunks to the vector store
    # $CHALLENGIFY_BEGIN
    document_ids = vector_store.add_documents(documents=all_splits)
    print(f"Added {len(document_ids)} documents to the vector store.")
    # $CHALLENGIFY_END

    return document_ids

👉 Fonksiyonunuzu bir dosya veya hatta iki dosya ile test edin:

In [ ]:
file_path = "data/CRE-10-2025-05-05_EN.pdf"

document_ids = embed_and_store(file_path, vector_store)

In [ ]:
vector_store.get_by_ids(document_ids[:3])

#### 3. `answer()` oluşturun

👉 Bu fonksiyon için kodu tamamlayın:

In [ ]:
def answer(query, vector_store, llm, prompt_template=None):
    """Answer a query using the vector store and the language model."""
    # Retrieve similar documents from the vector store
    retrieved_docs = vector_store.similarity_search(query, k=6)

    # Create the prompt
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # If no prompt template is provided, use the default one
    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # Get the answer from the language model
    answer = llm.invoke(prompt)

    return answer.content

👉 Fonksiyonunuzu beğendiğiniz bir sorgu ile test edin:

In [ ]:
query = "What is being said about international trade?"
Markdown(answer(query, vector_store, llm, prompt_template))

🏁 Tebrikler! Artık LangChain kullanarak RAG'ı ustalıkla kullanıyorsunuz ve vektör deponuza daha fazla döküman eklemek ve onu sorgulamak için yeniden kullanılabilir fonksiyonlar oluşturmayı öğrendiniz.

## [Opsiyonel] Metadata ekleme

Kurduğumuz RAG vektör deposundaki tüm dökümanları sorgular. Orada birkaç yıllık bilgimiz olduğunu hayal edin. Yıllara veya tarihlere göre filtreleyebilsek kullanışlı olurdu, değil mi?

Bunu nasıl yaparsınız? Vektör deposundaki dökümanların metadata içerdiğini unutmayın. Eğer buna tarihi ekleyebilseydik, daha sonra filtrelemek için kullanabiliriz.

İpucu: Metadata'nızı pipeline'ınızda mümkün olduğunca erken ekleyin. Verileriniz vektör deposunda zaten depolandıktan sonra eklemeye çalışmayın.

👉 `embed_and_store()` fonksiyonunuzu uyarlayın.

In [ ]:
def embed_and_store_fancy(file_path, vector_store, session_date):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store.
    Session_date is added to the metadata of each chunk."""
    # $CHALLENGIFY_BEGIN
    # Load the PDF file
    loader = PyPDFLoader(file_path, mode="single")
    pdf = loader.load()

    # Add the session_date to the metadata
    for doc in pdf:
        doc.metadata["session_date"] = session_date
        doc.metadata["year"] = session_date[:4]
        doc.metadata["month"] = session_date[:7]

    # Split the pages into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2_000,  # chunk size (characters)
        chunk_overlap=400,  # chunk overlap (characters)
        add_start_index=True,  # track index in original document
    )
    all_splits = text_splitter.split_documents(pdf)

    # Add the chunks to the vector store
    document_ids = vector_store.add_documents(documents=all_splits)
    print(f"Added {len(document_ids)} documents to the vector store.")
    # $CHALLENGIFY_END

    return document_ids

👉 Fonksiyonunuzu test edin ve vektör deponuzun ekstra metadata içerdiğini kontrol edin.

In [ ]:
file_path = "data/CRE-10-2025-05-05_EN.pdf"
document_ids = embed_and_store_fancy(file_path, vector_store, "2025-05-05")

In [ ]:
vector_store.get_by_ids(document_ids[:3])

şimdi retriever'ı kullanıcı tarafından sorulan tarihle sınırlamamz gerekiyor. 

👉 `answer()` fonksiyonunuzu bir tarih alacak ve yeni metadata'ya dayalı olarak dökümanları filtreleyecek şekilde uyarlayın.

In [ ]:
def answer_fancy(query, vector_store, llm, prompt_template=None, session_date=None, session_year=None, session_month=None):
    """Answer a query using the vector store and the language model."""
    # Retrieve similar documents from the vector store
    if session_date:
        filter = {"session_date": session_date}
    elif session_month:
        filter = {"month": session_month}
    elif session_month:
        filter = {"year": session_year}
    else:
        filter = {}

    # Perform the similarity search with the filter
    retrieved_docs = vector_store.similarity_search(query, k=6, filter=filter)

    # If no documents are found, return a message
    if not retrieved_docs:
        return "No documents found for the given filters."

    # Create the prompt
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # If no prompt template is provided, use the default one
    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # Get the answer from the language model
    answer = llm.invoke(prompt)

    return answer.content

In [ ]:
query = "Summarize the discussion on agricultural policy."
Markdown(answer_fancy(query, vector_store, llm, prompt_template, session_date="2025-05-05"))

In [ ]:
Markdown(answer_fancy(query, vector_store, llm, prompt_template, session_date="2025-05-06"))

In [ ]:
Markdown(answer_fancy(query, vector_store, llm, prompt_template, session_month="2025-05"))

Harika! Güçlü bir RAG sistemi oluşturmak için benzerlik aramasını metadata aramasıyla birleştirdiniz!